# 1. Creating Table

## 1.1 Adding Constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.fct_arrears_suburb(
    arrears_key BIGINT NOT NULL,
    date_key INT NOT NULL,
    suburb_key BIGINT NOT NULL,
    service_name STRING,
    ageing_bucket_days STRING NOT NULL,
    amount DECIMAL(13,2),

    CONSTRAINT pk_fct_arrears_suburb PRIMARY KEY(arrears_key) RELY,
    CONSTRAINT fk_fct_arrears_suburb_date FOREIGN KEY(date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY,
    CONSTRAINT fk_fct_arrears_suburb_suburb FOREIGN KEY (suburb_key) REFERENCES cpt_utility_catalog.gold.dim_suburb(suburb_key) RELY

)

## 1.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.fct_arrears_suburb
SELECT
xxhash64(s.id) AS arrears_key,

COALESCE(CAST(date_format(s.date, 'yyyyMMdd') AS INT), -1) AS date_key,
COALESCE(ds.suburb_key, xxhash64('unmapped')) AS suburb_key,

INITCAP(s.service_name) AS service_name,
s.ageing_bucket_days,
COALESCE(s.amount, 0.00) AS amount

FROM cpt_utility_catalog.silver.silver_suburb_arrears_cleaned s

LEFT JOIN cpt_utility_catalog.gold.dim_suburb ds
ON xxhash64(LOWER(TRIM(s.suburb))) = ds.suburb_key